In [8]:
import requests
from datetime import datetime, timedelta, timezone


# 使用 Gamma API 獲取市場資料
url = "https://gamma-api.polymarket.com/markets"

# 為了確保能找到目標，我們稍微拉高抓取數量 (limit)
# 在實際應用中，如果資料量很大，可能需要撰寫迴圈處理分頁 (offset)
params = {
    "active": "true",
    "closed": "false",
    "limit": 1500  
}

try:
    response = requests.get(url, params=params)
    response.raise_for_status()
    markets = response.json()
    
    # 設定時間邊界：現在時間與 2 天後 (Polymarket 使用 UTC 時間)
    now_utc = datetime.now(timezone.utc)
    two_days_later = now_utc + timedelta(days=7)
    
    print(f"=== 尋找 2 天內結束的 Soccer 市場 ===")
    print(f"現在時間 (UTC): {now_utc.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"目標時間 (UTC): {two_days_later.strftime('%Y-%m-%d %H:%M:%S')}\n")
    
    found_markets = []
    
    for market in markets:
        # 1. 篩選主題：檢查標籤 (tags)、分類 (category) 或標題 (question) 中是否有 Soccer
        # 將字串都轉為小寫方便比對
        tags = [tag.lower() for tag in market.get('tags', [])]
        category = market.get('category', '').lower()
        title = market.get('question', '').lower()
        
        # 有些足球活動標籤可能是 football，這裡一併加入檢查
        # 擴充足球相關的關鍵字
        soccer_keywords = ['soccer', 'football', 'epl', 'champions league', 'premier league', 'fifa']

        # 檢查是否有任何關鍵字出現在標籤、分類或標題中
        is_soccer = any(kw in tags for kw in soccer_keywords) or \
                    any(kw in category for kw in soccer_keywords) or \
                    any(kw in title for kw in soccer_keywords)
        
        if not is_soccer:
            continue
            
        # 2. 篩選結束時間：解析 endDate
        end_date_str = market.get('endDate')
        if not end_date_str:
            continue
            
        try:
            # Polymarket 的時間格式通常為 ISO 8601 (例: 2024-03-05T12:00:00Z)
            # Python 解析前將 'Z' 替換為 '+00:00' 確保加上 UTC 時區資訊
            end_date = datetime.fromisoformat(end_date_str.replace('Z', '+00:00'))
            
            # 判斷結束時間是否在現在與 2 天後之間
            if now_utc < end_date <= two_days_later:
                found_markets.append((market, end_date))
        except ValueError:
            # 如果遇到時間格式無法解析的問題，直接跳過該筆資料
            continue
    
    # 3. 輸出結果
    if not found_markets:
        print("目前沒有在 2 天內結束的 Soccer 市場。")
    else:
        # 依據結束時間排序 (最快結束的排在最前面)
        found_markets.sort(key=lambda x: x[1])
        
        for market, end_date in found_markets:
            print(f"問題: {market.get('question')}")
            print(f"市場 ID: {market.get('id')}")
            print(f"結束時間: {end_date.strftime('%Y-%m-%d %H:%M:%S')} (UTC)")
            print(f"倒數時間: {end_date - now_utc}")
            print(f"目前交易量: ${market.get('volume', '0')}")
            print("-" * 45)
            
except requests.exceptions.RequestException as e:
    print(f"API 請求失敗: {e}")



=== 尋找 2 天內結束的 Soccer 市場 ===
現在時間 (UTC): 2026-03-02 16:47:20
目標時間 (UTC): 2026-03-09 16:47:20

目前沒有在 2 天內結束的 Soccer 市場。


In [11]:
import requests

url = "https://clob.polymarket.com/events"
params = {
    "category": "Serie A",
    "limit": 100  # 一次最多抓 100 筆
}

response = requests.get(url, params=params)

if response.status_code == 200:
    events = response.json()
    for event in events:
        print(event["title"])
else:
    print("Error:", response.status_code)

Error: 404


In [17]:
import requests
import json

def get_serie_a_events():
    url = "https://gamma-api.polymarket.com/events"
    
    params = {
        "active": "true",
        "closed": "false",
        "limit": 1000
    }
    
    print("=== 尋找 Serie A (義大利甲級足球聯賽) 體育賽事 ===\n")
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        events = response.json()
        
        found_events = []
        
        for event in events:
            # --- 修正的地方：處理 tags 字典陣列 ---
            tags = []
            for tag_obj in event.get('tags', []):
                # 檢查如果是字典，就抓出裡面的 label 或 slug
                if isinstance(tag_obj, dict):
                    tag_str = tag_obj.get('label', '') or tag_obj.get('slug', '')
                    tags.append(str(tag_str).lower())
                # 如果是字串就直接用
                elif isinstance(tag_obj, str):
                    tags.append(tag_obj.lower())
            
            title = event.get('title', '').lower()
            slug = event.get('slug', '').lower() # 例如：sea-pis-bol-2026-03-02
            
            # 篩選條件：標題、標籤或 slug 包含 'serie a' 或 'serie-a'
            if 'serie a' in tags or 'serie a' in title or 'serie-a' in slug:
                found_events.append(event)
                
        if not found_events:
            print("目前沒有找到活躍的 Serie A 賽事。")
            return
            
        # 輸出找到的賽事
        for event in found_events:
            print(f"⚽ 賽事名稱: {event.get('title')}")
            print(f"   賽事 ID: {event.get('id')}")
            print(f"   網址 Slug: {event.get('slug')}")
            
            markets = event.get('markets', [])
            print(f"   包含市場數: {len(markets)}")
            
            for market in markets:
                print(f"\n   -> 市場問題: {market.get('question')}")
                print(f"      市場 ID: {market.get('id')}")
                
                outcomes = market.get('outcomes', '[]')
                prices = market.get('outcomePrices', '[]')
                
                if isinstance(outcomes, str): outcomes = json.loads(outcomes)
                if isinstance(prices, str): prices = json.loads(prices)
                
                print("      [當前即時機率]")
                for outcome, price in zip(outcomes, prices):
                    try:
                        prob = float(price) * 100
                        print(f"        - {outcome}: {prob:.1f}% (${price})")
                    except (ValueError, TypeError):
                        print(f"        - {outcome}: 無法解析")
            print("-" * 60)
            
    except requests.exceptions.RequestException as e:
        print(f"API 請求失敗: {e}")

get_serie_a_events()

=== 尋找 Serie A (義大利甲級足球聯賽) 體育賽事 ===

⚽ 賽事名稱: Serie A League Winner 
   賽事 ID: 33659
   網址 Slug: serie-a-league-winner
   包含市場數: 25

   -> 市場問題: Will Roma win the 2025–26 Serie A league?
      市場 ID: 566515
      [當前即時機率]
        - Yes: 0.5% ($0.0055)
        - No: 99.5% ($0.9945)

   -> 市場問題: Will Atalanta win the 2025–26 Serie A league?
      市場 ID: 566516
      [當前即時機率]
        - Yes: 0.1% ($0.0005)
        - No: 100.0% ($0.9995)

   -> 市場問題: Will Lazio win the 2025–26 Serie A league?
      市場 ID: 566517
      [當前即時機率]
        - Yes: 0.1% ($0.0005)
        - No: 100.0% ($0.9995)

   -> 市場問題: Will Fiorentina win the 2025–26 Serie A league?
      市場 ID: 566518
      [當前即時機率]
        - Yes: 0.0% ($0)
        - No: 100.0% ($1)

   -> 市場問題: Will Bologna win the 2025–26 Serie A league?
      市場 ID: 566519
      [當前即時機率]
        - Yes: 0.2% ($0.002)
        - No: 99.8% ($0.998)

   -> 市場問題: Will Como win the 2025–26 Serie A league?
      市場 ID: 566520
      [當前即時機率]
        - Yes: 0.1% ($0.

In [24]:
import requests
from collections import Counter

def get_active_soccer_leagues():
    url = "https://gamma-api.polymarket.com/events"
    
    # 抓取最新的活躍事件，limit 設為 1000 通常足以涵蓋大部分體育賽事
    params = {
        "active": "true",
        "closed": "false",
        "limit": 1000
    }
    
    print("⚽ 正在掃描 Polymarket，尋找目前活躍的足球聯賽 (Soccer Leagues)...\n")
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        events = response.json()
        
        # 用來計算各個聯賽標籤出現的次數
        league_counter = Counter()
        
        # 定義足球的基礎關鍵字
        soccer_keywords = ['soccer', 'football']
        
        for event in events:
            raw_tags = event.get('tags', [])
            tag_labels = []
            
            # 1. 將 tags 解析為乾淨的文字清單
            for tag in raw_tags:
                if isinstance(tag, dict):
                    label = tag.get('label', '')
                    if label:
                        tag_labels.append(label)
                elif isinstance(tag, str):
                    tag_labels.append(tag)
                    
            # 2. 判斷這場賽事是否屬於足球
            # 只要標籤中包含 soccer 或 football (不分大小寫)，就認定為足球賽事
            is_soccer = any(
                kw in label.lower() 
                for kw in soccer_keywords 
                for label in tag_labels
            )
            
            # 3. 如果是足球賽事，就把「除了 Sports 和 Soccer 以外」的標籤收集起來
            if is_soccer:
                for label in tag_labels:
                    lower_label = label.lower()
                    # 過濾掉太過寬泛的母分類標籤
                    if lower_label not in ['sports', 'soccer', 'football']:
                        league_counter[label] += 1
                        
        # --- 輸出結果 ---
        if not league_counter:
            print("目前沒有找到活躍的足球賽事標籤。")
            return
            
        print("✅ 找到以下足球相關的標籤 (聯賽/盃賽/球隊)，以及目前對應的活躍事件數量：\n")
        
        # 使用 most_common() 依照賽事數量由多到少排序
        for league, count in league_counter.most_common():
            print(f"  🏆 {league:<25} : {count} 場賽事")
            
        print("\n" + "-" * 50)
        print("💡 提示：Polymarket 的標籤除了聯賽名稱 (如 Premier League, La Liga)，有時也會包含熱門球隊的名稱。")
            
    except requests.exceptions.RequestException as e:
        print(f"API 請求失敗: {e}")

get_active_soccer_leagues()

⚽ 正在掃描 Polymarket，尋找目前活躍的足球聯賽 (Soccer Leagues)...

✅ 找到以下足球相關的標籤 (聯賽/盃賽/球隊)，以及目前對應的活躍事件數量：

  🏆 EPL                       : 7 場賽事
  🏆 La Liga                   : 4 場賽事
  🏆 Serie A                   : 4 場賽事
  🏆 bundesliga                : 4 場賽事
  🏆 Ligue 1                   : 4 場賽事
  🏆 FIFA World Cup            : 3 場賽事
  🏆 Champions League          : 2 場賽事
  🏆 Culture                   : 2 場賽事
  🏆 NFL                       : 2 場賽事
  🏆 world cup                 : 1 場賽事
  🏆 2026 FIFA World Cup       : 1 場賽事
  🏆 Hide From New             : 1 場賽事
  🏆 UCL                       : 1 場賽事
  🏆 UEFA Europa League        : 1 場賽事
  🏆 Europa League             : 1 場賽事
  🏆 NCAA Football             : 1 場賽事
  🏆 CFB                       : 1 場賽事
  🏆 NFL Draft                 : 1 場賽事
  🏆 EFL Cup                   : 1 場賽事
  🏆 Carabao Cup               : 1 場賽事
  🏆 Turkey                    : 1 場賽事
  🏆 World                     : 1 場賽事
  🏆 Celebrities               : 1 場賽事
  🏆 Trump                     : 1 

In [25]:
import requests
import json

def get_epl_events():
    url = "https://gamma-api.polymarket.com/events"
    
    # 抓取活躍賽事
    params = {
        "active": "true",
        "closed": "false",
        "limit": 1000
    }
    
    print("⚽ 正在尋找英格蘭超級聯賽 (EPL / Premier League) 的最新賽事與賠率...\n")
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        events = response.json()
        
        found_events = []
        
        for event in events:
            tags = []
            # 解析標籤
            for tag in event.get('tags', []):
                if isinstance(tag, dict):
                    tags.append(tag.get('label', '').lower())
                    tags.append(tag.get('slug', '').lower())
                elif isinstance(tag, str):
                    tags.append(tag.lower())
            
            title = event.get('title', '').lower()
            
            # 判斷條件：標籤或標題中包含 'epl' 或 'premier league'
            is_epl = any(kw in tag for tag in tags for kw in ['epl', 'premier league']) or \
                     'epl' in title or 'premier league' in title
                     
            if is_epl:
                found_events.append(event)
                
        if not found_events:
            print("目前沒有找到活躍的 EPL 賽事。可能是剛好沒有比賽，或是 API 標籤有變動。")
            return
            
        print(f"✅ 總共找到 {len(found_events)} 場 EPL 賽事！\n")
        print("=" * 60)
        
        # 輸出賽事與各市場的即時機率
        for event in found_events:
            print(f"🏆 賽事名稱: {event.get('title')}")
            print(f"   賽事 ID: {event.get('id')}")
            
            markets = event.get('markets', [])
            
            for market in markets:
                # 每個 Event 裡面可能有「誰會贏」、「會不會平手」等多個市場
                print(f"\n   -> 市場問題: {market.get('question')}")
                print(f"      市場 ID: {market.get('id')}")
                
                outcomes = market.get('outcomes', '[]')
                prices = market.get('outcomePrices', '[]')
                
                # 安全轉換 JSON 字串
                if isinstance(outcomes, str): outcomes = json.loads(outcomes)
                if isinstance(prices, str): prices = json.loads(prices)
                
                print("      [當前市場預測機率]")
                for outcome, price in zip(outcomes, prices):
                    try:
                        prob = float(price) * 100
                        print(f"        - {outcome}: {prob:.1f}% (${price})")
                    except (ValueError, TypeError):
                        pass
            print("-" * 60)
            
    except requests.exceptions.RequestException as e:
        print(f"API 請求失敗: {e}")

get_epl_events()

⚽ 正在尋找英格蘭超級聯賽 (EPL / Premier League) 的最新賽事與賠率...

✅ 總共找到 8 場 EPL 賽事！

🏆 賽事名稱: English Premier League Winner 
   賽事 ID: 33507

   -> 市場問題: Will Brentford win the 2025–26 English Premier League?
      市場 ID: 566201
      [當前市場預測機率]
        - Yes: 0.1% ($0.0005)
        - No: 100.0% ($0.9995)

   -> 市場問題: Will Newcastle win the 2025–26 English Premier League?
      市場 ID: 566190
      [當前市場預測機率]
        - Yes: 0.1% ($0.0005)
        - No: 100.0% ($0.9995)

   -> 市場問題: Will Club A win the 2025–26 English Premier League?
      市場 ID: 566207
      [當前市場預測機率]

   -> 市場問題: Will Crystal Palace win the 2025–26 English Premier League?
      市場 ID: 566199
      [當前市場預測機率]
        - Yes: 0.1% ($0.0005)
        - No: 100.0% ($0.9995)

   -> 市場問題: Will Brighton win the 2025–26 English Premier League?
      市場 ID: 566194
      [當前市場預測機率]
        - Yes: 0.1% ($0.0005)
        - No: 100.0% ($0.9995)

   -> 市場問題: Will Nottm Forest win the 2025–26 English Premier League?
      市場 ID: 566195
      [當前市場預測機

In [28]:
import requests
import json

def get_specific_event_data():
    # 從網址中提取出來的 slug
    target_slug = "lal-rea-get-2026-03-02"
    
    url = "https://gamma-api.polymarket.com/events"
    
    # 直接使用 slug 作為過濾參數，API 只會回傳這一場比賽
    params = {
        "slug": target_slug
    }
    
    print(f"🔍 正在獲取賽事 (Slug: {target_slug}) 的全部資料...\n")
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        events = response.json()
        
        if not events:
            print("❌ 找不到該賽事，可能是網址有誤或賽事已被隱藏。")
            return
            
        # 因為 slug 是唯一的，陣列中只會有一個目標事件
        target_event = events[0]
        
        # ==========================================
        # 1. 輸出賽事層級 (Event) 的基本資訊
        # ==========================================
        print("=== ⚽ 賽事基本資料 (Event Info) ===")
        print(f"📌 名稱: {target_event.get('title')}")
        print(f"🆔 賽事 ID: {target_event.get('id')}")
        print(f"⏰ 結束時間: {target_event.get('endDate')}")
        
        # 處理標籤 (有時是字串，有時是字典)
        tags = []
        for tag in target_event.get('tags', []):
            if isinstance(tag, dict):
                tags.append(tag.get('label', ''))
            else:
                tags.append(tag)
        print(f"🏷️ 標籤: {', '.join(tags)}")
        
        # ==========================================
        # 2. 輸出該賽事底下「所有」的市場 (Markets)
        # ==========================================
        markets = target_event.get('markets', [])
        print(f"\n=== 📊 包含的市場清單 (共 {len(markets)} 個玩法) ===")
        
        for i, market in enumerate(markets, 1):
            print(f"\n[{i}] 玩法問題: {market.get('question')}")
            print(f"    市場 ID: {market.get('id')}")
            print(f"    交易量: ${market.get('volume', '0')}")
            print(f"    狀態: {'已關閉' if market.get('closed') else '開放中'}")
            
            outcomes = market.get('outcomes', '[]')
            prices = market.get('outcomePrices', '[]')
            
            # 解析 JSON 字串
            if isinstance(outcomes, str): outcomes = json.loads(outcomes)
            if isinstance(prices, str): prices = json.loads(prices)
            
            print("    [選項與當前最新成交價]")
            for outcome, price in zip(outcomes, prices):
                try:
                    prob = float(price) * 100
                    print(f"      - {outcome}: {prob:.1f}% (${price})")
                except (ValueError, TypeError):
                    print(f"      - {outcome}: {price}")
                    
        # ==========================================
        # 3. (進階) 將最原始的完整 JSON 存成檔案
        # ==========================================
        # 如果你想看 API 到底還給了哪些隱藏欄位 (例如圖片網址、合約地址等)
        with open('event_full_data.json', 'w', encoding='utf-8') as f:
            json.dump(target_event, f, ensure_ascii=False, indent=4)
        print("\n" + "="*50)
        print("✅ 程式執行完畢！最完整的原始 JSON 資料已儲存到 'event_full_data.json'，你可以打開檔案查看所有隱藏欄位。")
            
    except requests.exceptions.RequestException as e:
        print(f"API 請求失敗: {e}")

get_specific_event_data()

🔍 正在獲取賽事 (Slug: lal-rea-get-2026-03-02) 的全部資料...

=== ⚽ 賽事基本資料 (Event Info) ===
📌 名稱: Real Madrid CF vs. Getafe CF
🆔 賽事 ID: 212550
⏰ 結束時間: 2026-03-02T20:00:00Z
🏷️ 標籤: Sports, La Liga, Games, Soccer

=== 📊 包含的市場清單 (共 3 個玩法) ===

[1] 玩法問題: Will Real Madrid CF win on 2026-03-02?
    市場 ID: 1388016
    交易量: $1216857.262624
    狀態: 開放中
    [選項與當前最新成交價]
      - Yes: 71.5% ($0.715)
      - No: 28.5% ($0.285)

[2] 玩法問題: Will Real Madrid CF vs. Getafe CF end in a draw?
    市場 ID: 1388018
    交易量: $77966.042687
    狀態: 開放中
    [選項與當前最新成交價]
      - Yes: 18.5% ($0.185)
      - No: 81.5% ($0.815)

[3] 玩法問題: Will Getafe CF win on 2026-03-02?
    市場 ID: 1388020
    交易量: $155884.960051
    狀態: 開放中
    [選項與當前最新成交價]
      - Yes: 9.5% ($0.095)
      - No: 90.5% ($0.905)

✅ 程式執行完畢！最完整的原始 JSON 資料已儲存到 'event_full_data.json'，你可以打開檔案查看所有隱藏欄位。


In [26]:
import requests

def get_events_with_specific_tags():
    url = "https://gamma-api.polymarket.com/events"
    
    # 這是我們規定「必須同時存在」的 4 個 Tag ID
    # 統一使用字串格式，避免 JSON 解析時型別不一致的問題
    required_tag_ids = {"1", "100639", "100350"}
    
    # 為了節省頻寬與時間，我們先讓 API 幫我們挑出含有 La Liga (780) 的賽事
    params = {
        "active": "true",
        "closed": "false",
        "tag_id": "100350", 
        "limit": 1000
    }
    
    print(" 正在尋找同時包含 [Sports, Games, Soccer] 這 4 個標籤的賽事...\n")
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        events = response.json()
        
        found_events = []
        
        for event in events:
            # 收集這場賽事「身上掛的所有 Tag ID」
            event_tag_ids = set()
            for tag in event.get('tags', []):
                if isinstance(tag, dict):
                    # 將 ID 轉為字串並加入集合中
                    event_tag_ids.add(str(tag.get('id')))
            
            # 核心邏輯：檢查 required_tag_ids 是否為 event_tag_ids 的「子集」
            # 意思是：這 4 個 ID 是不是「全部都在」這場比賽的標籤清單裡？
            if required_tag_ids.issubset(event_tag_ids):
                found_events.append(event)
                
        if not found_events:
            print("目前沒有找到同時完美包含這 4 個標籤的賽事。")
            return
            
        print(f" 總共找到 {len(found_events)} 場符合條件的賽事！\n")
        print("-" * 60)
        
        # 輸出結果，並順便把該賽事的所有標籤印出來當作驗證
        for i, event in enumerate(found_events, 1):
            print(f"{i}.  賽事名稱: {event.get('title')}")
            print(f"   賽事 ID: {event.get('id')}")
            print(f"   網址 Slug: {event.get('slug')}")
            
            # 把這場比賽掛的標籤名稱整理出來，方便我們肉眼檢查
            actual_labels = [t.get('label') for t in event.get('tags', []) if isinstance(t, dict)]
            print(f"   實際掛載的標籤: {', '.join(actual_labels)}")
            print("-" * 60)
            
    except requests.exceptions.RequestException as e:
        print(f"API 請求失敗: {e}")

get_events_with_specific_tags()

 正在尋找同時包含 [Sports, Games, Soccer] 這 4 個標籤的賽事...

 總共找到 401 場符合條件的賽事！

------------------------------------------------------------
1.  賽事名稱: Portsmouth FC vs. Ipswich Town FC
   賽事 ID: 99714
   網址 Slug: elc-por-ips-2026-01-04
   實際掛載的標籤: Sports, EFL Championship, Soccer, Games
------------------------------------------------------------
2.  賽事名稱: Sydney FC vs. Auckland FC - More Markets
   賽事 ID: 113849
   網址 Slug: aus-syd-auc-2025-12-27-more-markets
   實際掛載的標籤: Sports, Australian A-League, Games, Soccer
------------------------------------------------------------
3.  賽事名稱: Göztepe SK vs. Çaykur Rizespor
   賽事 ID: 119654
   網址 Slug: tur-goz-riz-2026-01-19
   實際掛載的標籤: Sports, Games, Soccer, Süper Lig
------------------------------------------------------------
4.  賽事名稱: Portsmouth FC vs. Ipswich Town FC - More Markets
   賽事 ID: 132906
   網址 Slug: elc-por-ips-2026-01-04-more-markets
   實際掛載的標籤: Sports, EFL Championship, Soccer, Games
------------------------------------------------------

In [4]:
import requests

def test_sx_bet_api():
    # SX Bet API 基礎網址
    base_url = "https://api.sx.bet"
    
    # 測試端點：獲取所有活躍的市場 (Active Markets)
    # 根據官方文件，這個端點通常不需要 API Key 即可讀取公開賠率
    endpoint = f"{base_url}/active-markets"
    
    print(f"正在連線至: {endpoint} ...")
    
    try:
        headers = {
            "X-Api-Key": "c75330b7-7ad6-460f-9e2d-cf7a26d9b917"
        }
        response = requests.get("https://api.sx.bet/user/token", headers=headers)
        
        
        # 檢查 HTTP 狀態碼
        if response.status_code == 200:
            data = response.json()
            # 假設回傳格式為 { "status": "success", "data": [...] }
            markets = data.get('data', [])
            
            print("✅ 成功連線！")
            print(f"目前活躍市場總數: {len(markets)}")
            
            # 印出前 3 個市場作為範例
            if markets:
                print("\n--- 市場範例 ---")
                for market in markets[:3]:
                    team1 = market.get('teamOneName', 'N/A')
                    team2 = market.get('teamTwoName', 'N/A')
                    league = market.get('leagueLabel', '未知聯盟')
                    print(f"聯盟: {league} | 賽事: {team1} vs {team2}")
            else:
                print("目前沒有活躍市場。")
                
        else:
            print(f"❌ 連線失敗，狀態碼: {response.status_code}")
            print(f"錯誤訊息: {response.text}")
            
    except Exception as e:
        print(f"發生異常: {e}")

if __name__ == "__main__":
    test_sx_bet_api()

正在連線至: https://api.sx.bet/active-markets ...
✅ 成功連線！
目前活躍市場總數: 0
目前沒有活躍市場。


In [13]:
import requests
market_=[]
def debug_sx_api():
    base_url = "https://api.sx.bet"
    
    # 請替換成你真實的 API Key
    headers = {
        "X-Api-Key": "c75330b7-7ad6-460f-9e2d-cf7a26d9b917",
        "Accept": "application/json"
    }

    print("=== 測試 1: 獲取 WebSocket Token ===")
    token_url = f"{base_url}/user/token"
    token_res = requests.get(token_url, headers=headers)
    print(f"狀態碼: {token_res.status_code}")
    
    if token_res.status_code == 200:
        print("回傳內容:", token_res.text) 
        print("💡 說明: 上面這串就是 WebSocket 需要的驗證資料，不是比賽數據。\n")
    else:
        print(f"錯誤: {token_res.text}\n")


    print("=== 測試 2: 獲取活躍市場 (Active Markets) ===")
    # 注意：SX Bet 官方的市場路徑通常為 /markets/active
    markets_url = f"{base_url}/markets/active"
    markets_res = requests.get(markets_url, headers=headers)
    print(f"狀態碼: {markets_res.status_code}")

    if markets_res.status_code == 200:
        data = markets_res.json()
        
        # 處理 SX Bet 回傳的資料結構 (可能是 dict 包含 'data'，也可能是直接回傳 list)
        markets = data.get('data', []) if isinstance(data, dict) else data
        
        if markets:
            print(f"✅ 成功抓到 {len(markets)} 筆市場數據！")
            print("--- 第一筆市場資料預覽 ---")
            # 只印出第一筆的前幾個重要欄位，避免畫面太亂
            first_market = markets
            market_=markets
        else:
            print("✅ 請求成功，但目前剛好沒有活躍的市場 (陣列為空)。")
    else:
         print(f"❌ 錯誤: {markets_res.text}")

if __name__ == "__main__":
    debug_sx_api()

=== 測試 1: 獲取 WebSocket Token ===
狀態碼: 200
回傳內容: {"keyName":"Pb_c6A.2IZKTQ","clientId":"0xd29956C15eE222626714F891C054982D6D36CDd6","capability":"{\"*\":[\"subscribe\"]}","ttl":86400000,"timestamp":1772727013190,"nonce":"4685773804906479","mac":"11Od1c9OK68KlMkLxg9DNKMaH+RRlL1+nIN6LzXb9yo="}
💡 說明: 上面這串就是 WebSocket 需要的驗證資料，不是比賽數據。

=== 測試 2: 獲取活躍市場 (Active Markets) ===
狀態碼: 200
✅ 成功抓到 2 筆市場數據！
--- 第一筆市場資料預覽 ---


In [15]:
import requests

def get_available_sports():
    base_url = "https://api.sx.bet"
    # 如果有 API Key 可以帶上，但這個端點通常是公開的
    headers = {
        "Accept": "application/json"
    }

    # 官方文件中 Sports 的端點
    sports_endpoint = f"{base_url}/sports"
    
    print(f"🔍 正在獲取支援的運動項目: {sports_endpoint}")
    
    try:
        response = requests.get(sports_endpoint, headers=headers, timeout=10)
        
        if response.status_code == 200:
            result = response.json()
            # 根據官方文件，回傳格式為 { "status": "success", "data": [ ... ] }
            sports = result.get('data', [])
            
            if sports:
                print(f"✅ 成功獲取 {len(sports)} 種運動項目！\n")
                print(f"{'運動 ID (sportId)':<15} | {'運動名稱 (label)'}")
                print("-" * 40)
                
                for sport in sports:
                    sport_id = sport.get('sportId')
                    label = sport.get('label')
                    print(f"{sport_id:<17} | {label}")
            else:
                print("⚠️ 請求成功，但沒有回傳任何運動項目。")
                
        else:
            print(f"❌ 請求失敗，狀態碼: {response.status_code}")
            print(f"錯誤訊息: {response.text}")
            
    except Exception as e:
        print(f"⚠️ 發生錯誤: {e}")

if __name__ == "__main__":
    get_available_sports()

🔍 正在獲取支援的運動項目: https://api.sx.bet/sports
✅ 成功獲取 29 種運動項目！

運動 ID (sportId) | 運動名稱 (label)
----------------------------------------
1                 | Basketball
2                 | Hockey
3                 | Baseball
4                 | Golf
5                 | Soccer
6                 | Tennis
7                 | Mixed Martial Arts
8                 | Football
9                 | E Sports
10                | Novelty Markets
11                | Rugby Union
12                | Racing
13                | Boxing
14                | Crypto
15                | Cricket
16                | Economics
17                | Politics
18                | Entertainment
19                | Medicinal
20                | Rugby League
21                | Olympics
22                | Athletics
23                | NFTs
24                | Horse Racing
25                | Daily Parlays
27                | Degen Crypto
28                | InplayLIVE
29                | Sailing
26                | AFL


In [31]:
import requests
import time
from datetime import datetime

def get_all_soccer_markets():
    base_url = "https://api.sx.bet"
    endpoint = f"{base_url}/markets/active"
    headers = {"Accept": "application/json"}
    
    # 準備一個空的清單，用來收集每一頁的比賽
    all_soccer_markets = []
    
    # 初始的下一頁鑰匙是空的
    next_key = None
    page_count = 1
    
    print("⚽ 開始向 SX Bet 請求所有足球 (Soccer) 活躍主盤口...\n")
    
    while True:
        # 基本參數
        params = {
            "sportIds": 5, 
            "onlyMainLine": "true",
            "pageSize": 50
        }
        
        # 如果有 next_key (代表要抓第二頁以後)，就把 paginationKey 塞進參數裡
        if next_key:
            params["paginationKey"] = next_key
            
        try:
            print(f"📄 正在抓取第 {page_count} 頁...")
            response = requests.get(endpoint, headers=headers, params=params, timeout=10)
            
            if response.status_code == 200:
                result = response.json()
                data = result.get('data', {})
                
                # 拿出這一頁的比賽
                current_page_markets = data.get('markets', [])
                all_soccer_markets.extend(current_page_markets) # 把這一頁的比賽裝進總清單
                
                # 更新 next_key
                next_key = data.get('nextKey')
                
                # 如果 next_key 是空的 (None 或空字串)，代表已經沒有下一頁了
                if not next_key:
                    print("🏁 已經抓取到最後一頁！")
                    break
                
                page_count += 1
                
                # 為了保護你的 IP 不被 API 伺服器因為「請求過快」而封鎖，稍微暫停一下
                time.sleep(0.5) 
                
            else:
                print(f"❌ 第 {page_count} 頁請求失敗，狀態碼: {response.status_code}")
                break
                
        except Exception as e:
            print(f"⚠️ 發生錯誤: {e}")
            break

    # --- 迴圈結束，開始印出總結果 ---
    if all_soccer_markets:
        print("\n" + "="*70)
        print(f"✅ 大功告成！總共抓取到 {len(all_soccer_markets)} 場足球賽事。")
        print("="*70)
        
        # 預覽前 5 筆跟最後 5 筆來確認資料
        print(f"\n--- 前 3 場比賽預覽 ---")
        for market in all_soccer_markets[:3]:
            print(f"{market.get('teamOneName')} vs {market.get('teamTwoName')} | {market.get('leagueLabel')}")
            
        print(f"\n--- 最後 3 場比賽預覽 ---")
        for market in all_soccer_markets[-3:]:
            print(f"{market.get('teamOneName')} vs {market.get('teamTwoName')} | {market.get('leagueLabel')}")
            
        print("\n💡 你現在擁有完整的足球比賽清單了！")
        # 挑選第一場比賽的 Hash 供下一步使用
        print(f"🔑 範例 Market Hash: {all_soccer_markets[0].get('marketHash')}")

if __name__ == "__main__":
    get_all_soccer_markets()

⚽ 開始向 SX Bet 請求所有足球 (Soccer) 活躍主盤口...

📄 正在抓取第 1 頁...
📄 正在抓取第 2 頁...
📄 正在抓取第 3 頁...
📄 正在抓取第 4 頁...
📄 正在抓取第 5 頁...
📄 正在抓取第 6 頁...
📄 正在抓取第 7 頁...
📄 正在抓取第 8 頁...
📄 正在抓取第 9 頁...
📄 正在抓取第 10 頁...
📄 正在抓取第 11 頁...
📄 正在抓取第 12 頁...
📄 正在抓取第 13 頁...
📄 正在抓取第 14 頁...
📄 正在抓取第 15 頁...
📄 正在抓取第 16 頁...
🏁 已經抓取到最後一頁！

✅ 大功告成！總共抓取到 713 場足球賽事。

--- 前 3 場比賽預覽 ---
Tottenham Hotspur vs Crystal Palace | English Premier League
Tottenham Hotspur vs Crystal Palace | English Premier League
Tottenham Hotspur vs Crystal Palace | English Premier League

--- 最後 3 場比賽預覽 ---
Deportes Tolima vs O'Higgins | Copa Libertadores
Deportes Tolima vs O'Higgins | Copa Libertadores
Deportes Tolima vs O'Higgins | Copa Libertadores

💡 你現在擁有完整的足球比賽清單了！
🔑 範例 Market Hash: 0xe7f7da29e219e56a153db71c0b47b9b0ea0717989f4ea0b6b5b86a8688a89e59


[{'status': 'ACTIVE',
  'marketHash': '0xe7f7da29e219e56a153db71c0b47b9b0ea0717989f4ea0b6b5b86a8688a89e59',
  'outcomeOneName': 'Tie',
  'outcomeTwoName': 'Not tie',
  'outcomeVoidName': 'NO_CONTEST',
  'teamOneName': 'Tottenham Hotspur',
  'teamTwoName': 'Crystal Palace',
  'type': 1,
  'gameTime': 1772740800,
  'sportXeventId': 'L18097153',
  'liveEnabled': True,
  'sportLabel': 'Soccer',
  'sportId': 5,
  'leagueId': 29,
  'leagueLabel': 'English Premier League',
  'group1': 'English Premier League',
  'chainVersion': 'SXR',
  'participantOneId': 21,
  'participantTwoId': 981,
  '__type': 'Market'},
 {'status': 'ACTIVE',
  'marketHash': '0xca204841b22e425990b7391226179d953786abeedb3cac94cb2105d789899dc8',
  'outcomeOneName': 'Tottenham Hotspur',
  'outcomeTwoName': 'Not Tottenham Hotspur',
  'outcomeVoidName': 'NO_CONTEST',
  'teamOneName': 'Tottenham Hotspur',
  'teamTwoName': 'Crystal Palace',
  'type': 1,
  'gameTime': 1772740800,
  'sportXeventId': 'L18097153',
  'liveEnabled': 

In [29]:
print(f"目前全站總活躍市場數: {len(all_markets)}")

目前全站總活躍市場數: 50


In [33]:
import requests

def get_best_odds_for_market(market_hash):
    endpoint = "https://api.sx.bet/orders"
    
    # 完全依照你貼的文件設定參數
    params = {
        "marketHashes": market_hash, # 注意這裡是複數
        "perPage": 1000              # 一次最多拿 1000 筆掛單
    }
    
    print(f"📊 正在分析 Market Hash: {market_hash[:10]}... 的盤口\n")
    
    try:
        response = requests.get(endpoint, params=params, timeout=10)
        
        if response.status_code == 200:
            result = response.json()
            # 根據文件，回傳的直接是 data 陣列
            orders = result.get('data', [])
            
            if not orders:
                print("📭 這場比賽目前沒有任何掛單。")
                return
            
            print(f"✅ 成功抓取到 {len(orders)} 筆掛單！正在計算最佳賠率...\n")
            
            # 用來儲存你要下注 選項1 或 選項2 的所有可用賠率
            odds_for_outcome_1 = []
            odds_for_outcome_2 = []
            
            for order in orders:
                # 排除已經被完全買光的單 (fillAmount == totalBetSize)
                if int(order.get('fillAmount', 0)) >= int(order.get('totalBetSize', 0)):
                    continue
                
                is_maker_outcome_one = order.get('isMakerBettingOutcomeOne')
                
                # 計算 Taker (你) 的實際歐洲賠率
                raw_odds = int(order.get('percentageOdds', 0))
                maker_implied_prob = raw_odds / (10 ** 20)
                taker_implied_prob = 1 - maker_implied_prob
                
                if taker_implied_prob <= 0:
                    continue
                    
                decimal_odds = 1 / taker_implied_prob
                
                # 分類：如果你要押選項 1，你要找 Maker 押選項 2 的單 (is_maker_outcome_one == False)
                if is_maker_outcome_one is False:
                    odds_for_outcome_1.append(decimal_odds)
                else:
                    # 如果你要押選項 2，你要找 Maker 押選項 1 的單
                    odds_for_outcome_2.append(decimal_odds)
            
            # 找出最高賠率
            best_odd_1 = max(odds_for_outcome_1) if odds_for_outcome_1 else 0
            best_odd_2 = max(odds_for_outcome_2) if odds_for_outcome_2 else 0
            
            print("🏆 目前市場最佳賠率 (Best Odds)：")
            print("-" * 40)
            print(f"如果你想下注【選項 1 (主隊)】: 最高賠率 {best_odd_1:.2f} 倍")
            print(f"如果你想下注【選項 2 (客隊)】: 最高賠率 {best_odd_2:.2f} 倍")
            print("-" * 40)

        else:
            print(f"❌ 請求失敗，狀態碼: {response.status_code}")
            print(response.text)
            
    except Exception as e:
        print(f"⚠️ 發生錯誤: {e}")

if __name__ == "__main__":
    # 使用你剛才的熱刺 vs 水晶宮的 Hash
    target_hash = "0x079d236524fe9b0978599414e51b5476c252368b9c83f82ae668f52359642953"
    get_best_odds_for_market(target_hash)

📊 正在分析 Market Hash: 0x079d2365... 的盤口

✅ 成功抓取到 8 筆掛單！正在計算最佳賠率...

🏆 目前市場最佳賠率 (Best Odds)：
----------------------------------------
如果你想下注【選項 1 (主隊)】: 最高賠率 1.76 倍
如果你想下注【選項 2 (客隊)】: 最高賠率 2.16 倍
----------------------------------------


In [11]:
import requests

response = requests.get(
    "https://api.sharpapi.io/api/v1/odds?league=Soccer",
    headers={"X-API-Key": "sk_live_RWRj9B83KMugBoPpSgFx4v"}
)
print(response.json())

{'success': True, 'data': [{'id': 183105831663243, 'sportsbook': 'draftkings', 'sportsbook_name': 'DraftKings', 'event_id': 'soccer_mfkkarvina_viktoriaplzen_2026-03-05', 'sport': 'soccer', 'league': 'soccer', 'home_team': 'MFK Karvina', 'away_team': 'Viktoria Plzen', 'market_type': 'asian_handicap', 'selection': 'MFK Karvina', 'selection_type': 'home', 'odds_american': 115, 'odds_decimal': 2.15, 'probability': 0.46511627906976744, 'line': 0.25, 'event_start_time': '2026-03-05T16:01:21Z', 'is_live': True, 'status': 'live', 'timestamp': '2026-03-05T17:31:03.198Z', 'home_score': 2, 'away_score': 0, 'game_period': '2H', 'game_clock': None, 'score_type': 'match', 'external_event_id': '33741865'}, {'id': 183105831663243, 'sportsbook': 'draftkings', 'sportsbook_name': 'DraftKings', 'event_id': 'soccer_mfkkarvina_viktoriaplzen_2026-03-05', 'sport': 'soccer', 'league': 'soccer', 'home_team': 'MFK Karvina', 'away_team': 'Viktoria Plzen', 'market_type': 'asian_handicap', 'selection': 'Viktoria Pl

In [ ]:
limitlessapi='lmts_Fdg2iohVwh9d7NvD_xgHRX-h8h42aAuMUn02VqEgKEpHm-Nxm1v9kp_8IwkE'

In [13]:
import requests
from datetime import datetime

def fetch_and_parse_limitless_markets():
    url = "https://api.limitless.exchange/markets/active"
    headers = {"Accept": "application/json"}
    
    print("🚀 正在從 Limitless 獲取活躍市場資料...\n")
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        
        if response.status_code == 200:
            result = response.json()
            markets = result.get('data', [])
            total_count = result.get('totalMarketsCount', 0)
            
            print(f"✅ 成功抓取！全站目前共有 {total_count} 個活躍市場。\n")
            print("=" * 85)
            
            # 這裡我們只印出前 3 筆做示範
            for market in markets[:3]:
                title = market.get('title', '未知標題')
                market_type = market.get('tradeType', '未知機制').upper()
                volume = market.get('volumeFormatted', '0')
                
                # 解析價格 (prices 陣列通常是 [YES價格, NO價格])
                prices = market.get('prices', [0, 0])
                yes_price = prices[0] if len(prices) > 0 else 0
                no_price = prices[1] if len(prices) > 1 else 0
                
                print(f"📌 市場: {title}")
                print(f"📊 機制: {market_type} | 💰 總交易量: {volume} USDC")
                print(f"💵 目前價格: [YES] {yes_price:.1f}¢  |  [NO] {no_price:.1f}¢")
                
                # 解析最新交易動態 (Feed Events)
                feed_events = market.get('feedEvents', [])
                if feed_events:
                    latest_event = feed_events[0]
                    event_data = latest_event.get('data', {})
                    user_name = latest_event.get('user', {}).get('name', '匿名玩家')
                    
                    strategy = event_data.get('strategy', '')
                    outcome = event_data.get('outcome', '')
                    trade_amount = event_data.get('tradeAmountUSD', '0')
                    
                    print(f"⚡ 最新動態: 玩家 {user_name} 剛剛花費了 ${float(trade_amount):.2f} 買入 {outcome} ({strategy})")
                
                print("-" * 85)
                
        else:
            print(f"❌ 請求失敗，狀態碼: {response.status_code}")
            
    except Exception as e:
        print(f"⚠️ 發生錯誤: {e}")

if __name__ == "__main__":
    fetch_and_parse_limitless_markets()

🚀 正在從 Limitless 獲取活躍市場資料...

✅ 成功抓取！全站目前共有 474 個活躍市場。

📌 市場: $SOL above $88.939 on Mar 5, 19:00 UTC?
📊 機制: CLOB | 💰 總交易量: 22758.793000 USDC
💵 目前價格: [YES] 0.2¢  |  [NO] 0.8¢
-------------------------------------------------------------------------------------
📌 市場: $DOGE above $0.09398 on Mar 5, 19:00 UTC?
📊 機制: CLOB | 💰 總交易量: 31668.892472 USDC
💵 目前價格: [YES] 0.2¢  |  [NO] 0.8¢
-------------------------------------------------------------------------------------
📌 市場: $XRP above $1.4132 on Mar 5, 19:00 UTC?
📊 機制: CLOB | 💰 總交易量: 42796.153000 USDC
💵 目前價格: [YES] 0.2¢  |  [NO] 0.8¢
-------------------------------------------------------------------------------------


In [15]:
import requests

slug = "coupedefra-lyon-vs-lens-mar-5-2026-1771953663390"
# 根據 API 文件，查詢單一市場的端點是 /markets/{slug}
url = f"https://api.limitless.exchange/markets/{slug}"
headers = {"Accept": "application/json"}

print(f"🔍 正在鎖定查詢目標...\n🔗 Slug: {slug}")

try:
    response = requests.get(url, headers=headers, timeout=10)
    
    if response.status_code == 200:
        result = response.json()
        
        # Limitless 的單一市場回傳可能包在 data 裡，也可能直接是物件
        market = result.get('data', result)
        
        if not market:
            print("📭 找不到這個市場，請確認網址 Slug 是否正確。")
            
        
        # --- 開始萃取重要情報 ---
        title = market.get('title', '未知標題')
        status = market.get('status', '未知狀態')
        trade_type = market.get('tradeType', '未知機制').upper()
        
        volume = float(market.get('volumeFormatted', 0))
        liquidity = float(market.get('liquidityFormatted', 0))
        
        # 價格解析 (代表發生的機率)
        prices = market.get('prices', [0, 0])
        yes_price = prices[0] if len(prices) > 0 else 0
        no_price = prices[1] if len(prices) > 1 else 0
        
        # --- 華麗印出報表 ---
        print("\n" + "=".center(60, "="))
        print(f"🏟️ 賽事名稱: {title}")
        print(f"📈 狀態: {status} | ⚙️ 機制: {trade_type}")
        print(f"💰 總交易量: {volume:,.2f} USDC")
        print(f"💧 資金池深度: {liquidity:,.2f} USDC")
        print("-".center(60, "-"))
        
        print(f"💵 目前市場定價 (發生機率):")
        print(f"   🟢 [YES] 買入價: {yes_price:.1f}¢ (若命中可贏得 1 USDC，淨賺 {(100-yes_price):.1f}¢)")
        print(f"   🔴 [NO]  買入價: {no_price:.1f}¢ (若命中可贏得 1 USDC，淨賺 {(100-no_price):.1f}¢)")
        print("=".center(60, "=") + "\n")
        
    else:
        print(f"❌ 查詢失敗，伺服器回傳狀態碼: {response.status_code}")
        print(f"錯誤細節: {response.text}")
        
except Exception as e:
    print(f"⚠️ 發生錯誤: {e}")


🔍 正在鎖定查詢目標...
🔗 Slug: coupedefra-lyon-vs-lens-mar-5-2026-1771953663390

🏟️ 賽事名稱: ⚽ CoupeDeFra, Lyon vs Lens, Mar 5, 2026
📈 狀態: FUNDED | ⚙️ 機制: CLOB
💰 總交易量: 194.78 USDC
💧 資金池深度: 0.00 USDC
------------------------------------------------------------
💵 目前市場定價 (發生機率):
   🟢 [YES] 買入價: 0.0¢ (若命中可贏得 1 USDC，淨賺 100.0¢)
   🔴 [NO]  買入價: 0.0¢ (若命中可贏得 1 USDC，淨賺 100.0¢)



In [16]:
result

{'automationType': 'sports',
 'id': 10001009,
 'slug': 'coupedefra-lyon-vs-lens-mar-5-2026-1771953663390',
 'outcomeTokens': ['Yes', 'No'],
 'title': '⚽ CoupeDeFra, Lyon vs Lens, Mar 5, 2026',
 'ogImageURI': '',
 'imageUrl': None,
 'expirationDate': 'Mar 6, 2026',
 'expirationTimestamp': 1772827800000,
 'expired': False,
 'creator': {'name': 'Limitless',
  'imageURI': 'https://limitless.exchange/assets/images/logo.svg',
  'link': 'https://x.com/trylimitless'},
 'collateralToken': {'symbol': 'USDC',
  'address': '0x833589fCD6eDb6E08f4c7C32D4f71b54bdA02913',
  'decimals': 6},
 'tags': ['Football', 'Coupe de France'],
 'tradeType': 'clob',
 'marketType': 'group',
 'createdAt': '2026-02-24T17:26:02.495Z',
 'updatedAt': '2026-03-01T11:00:00.672Z',
 'categories': ['Football Matches'],
 'status': 'FUNDED',
 'priorityIndex': 0,
 'negRiskMarketId': '0x7b8d051db6ff9a1921a5c0ca595454e442f8b80e3e73660f892959c3d6da4900',
 'markets': [{'id': 53445,
   'automationType': 'manual',
   'conditionId': '0

In [19]:
import requests

def get_limitless_categories():
    # 根據常見的 REST API 設計，分類通常獨立在一個端點
    url = "https://api.limitless.exchange/categories"
    
    headers = {
        "Accept": "application/json",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0"
    }

    print("📡 正在向 Limitless 請求完整的【分類目錄 (Categories)】...\n")
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        
        if response.status_code == 200:
            result = response.json()
            
            # 有些 API 回傳直接是陣列，有些包在 data 裡
            categories = result.get('data', result) if isinstance(result, dict) else result
            
            if not categories:
                print("📭 目前沒有獲取到任何分類。")
                return
                
            print(f"✅ 成功獲取 {len(categories)} 種分類！\n")
            print(f"{'分類 ID (categoryId)':<25} | {'分類名稱 (Name)'}")
            print("-" * 60)
            
            # 遍歷並印出所有分類及其 ID
            for cat in categories:
                c_id = cat.get('id', '未知 ID')
                c_name = cat.get('name', cat.get('title', '未知名稱'))
                
                print(f"{str(c_id):<25} | {c_name}")
                
            print("\n💡 提示：找出 'Football' 或 'Sports' 的 ID 後，就能把它加進過濾參數了！")
                
        else:
            print(f"❌ 請求失敗，狀態碼: {response.status_code}")
            print(response.text)
            
    except Exception as e:
        print(f"⚠️ 發生錯誤: {e}")

if __name__ == "__main__":
    get_limitless_categories()

📡 正在向 Limitless 請求完整的【分類目錄 (Categories)】...

✅ 成功獲取 37 種分類！

分類 ID (categoryId)        | 分類名稱 (Name)
------------------------------------------------------------
46                        | 5 min
45                        | 15 min
33                        | 30 min
44                        | Minutely
29                        | Hourly
28                        | Bitcoin
30                        | Daily
31                        | Weekly
49                        | Football Matches
50                        | Off the Pitch
1                         | Sports
53                        | Esports
52                        | Culture
48                        | This vs That
2                         | Crypto
21                        | Indexes
51                        | Politics
23                        | Economy
19                        | Company News
8                         | Financials
43                        | Pre-TGE
42                        | Korean Market
39                  

In [4]:
import requests
import time

def get_all_limitless_football_markets():
    url = "https://api.limitless.exchange/markets/active/49"
    headers = {
        "Accept": "application/json",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36"
    }
    
    all_markets = []
    limit = 25
    page = 1  # 🌟 這次我們直接用直觀的頁碼 (Page) 來翻頁
    
    print("⚽ 開始使用 `page` 參數自動翻頁抓取...\n")
    
    try:
        while True:
            # 改用 limit 和 page 組合
            params = {
                "limit": limit,
                "page": page
            }
            
            print(f"📄 正在抓取第 {page} 頁...")
            response = requests.get(url, headers=headers, params=params, timeout=10)
            
            if response.status_code == 200:
                result = response.json()
                current_markets = result.get('data', result) if isinstance(result, dict) else result
                
                # 如果這一頁沒有資料，代表已經抓完了
                if not current_markets:
                    print("🏁 這一頁為空，抓取完畢！")
                    break
                
                all_markets.extend(current_markets)
                print(f"   ✅ 成功抓到 {len(current_markets)} 筆資料。")
                
                # 如果這頁拿到的資料小於 25 筆，代表這是最後一頁了
                if len(current_markets) < limit:
                    print("🏁 已經到達最後一頁，抓取完畢！")
                    break
                
                # 準備抓下一頁
                page += 1
                
                # 遵守速率限制，暫停 0.3 秒
                time.sleep(0.3)
                
            else:
                # 萬一連 page 都不吃，我們要把錯誤印出來看
                print(f"❌ 第 {page} 頁請求失敗，狀態碼: {response.status_code}")
                print(f"錯誤訊息: {response.text}")
                break
                
    except Exception as e:
        print(f"⚠️ 發生錯誤: {e}")

    # --- 總結報表 ---
    if all_markets:
        print("\n" + "=" * 60)
        print(f"🏆 大功告成！全站總共抓取到 {len(all_markets)} 場足球比賽。")
        print("=" * 60)
        
        print("\n隨機預覽最後 2 場比賽：")
        for market in all_markets[-2:]:
            title = market.get('title', '未知賽事')
            slug = market.get('slug', 'N/A')
            print(f" - {title[:45]}... | 🔗 Slug: {slug}...")

if __name__ == "__main__":
    get_all_limitless_football_markets()

⚽ 開始使用 `page` 參數自動翻頁抓取...

📄 正在抓取第 1 頁...
   ✅ 成功抓到 25 筆資料。
📄 正在抓取第 2 頁...
   ✅ 成功抓到 25 筆資料。
📄 正在抓取第 3 頁...
   ✅ 成功抓到 25 筆資料。
📄 正在抓取第 4 頁...
   ✅ 成功抓到 25 筆資料。
📄 正在抓取第 5 頁...
   ✅ 成功抓到 25 筆資料。
📄 正在抓取第 6 頁...
   ✅ 成功抓到 21 筆資料。
🏁 已經到達最後一頁，抓取完畢！

🏆 大功告成！全站總共抓取到 146 場足球比賽。

隨機預覽最後 2 場比賽：
 - ⚽ LaLiga, Real Madrid vs Atletico Madrid, Mar... | 🔗 Slug: laliga-real-madrid-vs-atletico-madrid-mar-22-2026-1772960406162...
 - ⚽ EFL CUP, Arsenal vs Manchester City, Mar 22... | 🔗 Slug: efl-cup-arsenal-vs-manchester-city-mar-22-2026-1772450423048...


In [ ]:
1773491400

In [2]:
from datetime import datetime, timezone

start_time = datetime.fromtimestamp(int('1773491400'), tz=timezone.utc)

In [3]:
start_time

datetime.datetime(2026, 3, 14, 12, 30, tzinfo=datetime.timezone.utc)

In [9]:
import json
address_or_slug="bundl-borussia-dortmund-vs-fc-augsburg-mar-14-2026-1772269214880"
url = f"https://api.limitless.exchange/markets/{address_or_slug}"
    
# 加上 headers 避免被 API 擋下來
headers = {
    "Accept": "application/json",
    "User-Agent": "Mozilla/5.0"
}

print("="*60)
print(" 📡 [單點查詢] 正在呼叫 Limitless API...")
print(f" 🔍 目標 Address 或 Slug: {address_or_slug}")
print(f" 🔗 完整網址: {url}")
print("="*60)

try:
    # 發送 GET 請求
    response = requests.get(url, headers=headers, timeout=10)
    
    # 檢查是否發生 HTTP 錯誤 (例如 404 找不到)
    response.raise_for_status() 
    
    raw_data = response.json()
    
    # 為了避免檔名有奇怪的字元，我們做個簡單的清理
    safe_name = address_or_slug.replace("/", "_").replace("?", "_")
    filename = f"limitless_debug_{safe_name}.json"
    
    # 存成漂亮的 JSON 檔案
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(raw_data, f, indent=4, ensure_ascii=False)
        
    print(f"\n🎉 查詢成功！這包盤口的完整資料已儲存至：{filename}")
    print("💡 建議使用 VS Code 打開它，尋找賠率 (prices) 和資金池大小 (liquidity) 相關的欄位。")
    
except requests.exceptions.HTTPError as errh:
    print(f"\n❌ HTTP 錯誤 (可能是 Address/Slug 錯誤或市場已結算): {errh}")
    print(f"官方回傳內容: {response.text}")
except Exception as e:
    print(f"\n❌ 發生未知的錯誤: {e}")

 📡 [單點查詢] 正在呼叫 Limitless API...
 🔍 目標 Address 或 Slug: bundl-borussia-dortmund-vs-fc-augsburg-mar-14-2026-1772269214880
 🔗 完整網址: https://api.limitless.exchange/markets/bundl-borussia-dortmund-vs-fc-augsburg-mar-14-2026-1772269214880

🎉 查詢成功！這包盤口的完整資料已儲存至：limitless_debug_bundl-borussia-dortmund-vs-fc-augsburg-mar-14-2026-1772269214880.json
💡 建議使用 VS Code 打開它，尋找賠率 (prices) 和資金池大小 (liquidity) 相關的欄位。


In [8]:
raw_data 

{'automationType': 'sports',
 'id': 10001108,
 'slug': 'bundl-borussia-dortmund-vs-fc-augsburg-mar-14-2026-1772269214880',
 'outcomeTokens': ['Yes', 'No'],
 'title': '⚽ BUNDL, Borussia Dortmund vs FC Augsburg, Mar 14, 2026',
 'ogImageURI': '',
 'imageUrl': None,
 'expirationDate': 'Mar 15, 2026',
 'expirationTimestamp': 1773585000000,
 'expired': False,
 'creator': {'name': 'Limitless',
  'imageURI': 'https://limitless.exchange/assets/images/logo.svg',
  'link': 'https://x.com/trylimitless'},
 'collateralToken': {'symbol': 'USDC',
  'address': '0x833589fCD6eDb6E08f4c7C32D4f71b54bdA02913',
  'decimals': 6},
 'tags': ['Football', 'Bundesliga', 'club_dominance'],
 'tradeType': 'clob',
 'marketType': 'group',
 'createdAt': '2026-03-01T10:07:08.905Z',
 'updatedAt': '2026-03-10T11:00:01.273Z',
 'categories': ['Football Matches'],
 'status': 'FUNDED',
 'priorityIndex': 0,
 'negRiskMarketId': '0x363f6c7e5b96d0c5ff23ce89d2244632821ac4732518964a828a4d620fc38700',
 'markets': [{'id': 55316,
   'a

In [14]:
from rapidfuzz import fuzz, process

def test_team_matching():
    # 假設這是你資料庫裡標準的球隊名單
    canonical_teams = [
        "independiente_medellín_juventud_de_las_piedras_2026-03-13",
        "independiente_medellín_juventud_de_las_piedras_2026-03-13"
    ]

    # 這是從 Polymarket 或 Limitless 抓下來的各種「髒名字」
    dirty_names = [
        "independiente_medellín_ca_juventud_2026-03-13",
        "independiente_medellín_cdp_junior_2026-03-20"
    ]

    print("="*50)
    print(" 🔍 模糊比對測試開始 (使用 WRatio)")
    print("="*50)

    for dirty in dirty_names:
        # process.extractOne 會從清單中挑出「最高分」的那一個
        best_match = process.extractOne(dirty, canonical_teams, scorer=fuzz.WRatio)
        
        match_name = best_match[0]
        score = best_match[1]
        
        print(f"髒名字: {dirty:<15} ➡️ 預測: {match_name:<18} | 相似度: {score:.1f} 分")

if __name__ == "__main__":
    test_team_matching()

 🔍 模糊比對測試開始 (使用 WRatio)
髒名字: independiente_medellín_ca_juventud_2026-03-13 ➡️ 預測: independiente_medellín_juventud_de_las_piedras_2026-03-13 | 相似度: 82.4 分
髒名字: independiente_medellín_cdp_junior_2026-03-20 ➡️ 預測: independiente_medellín_juventud_de_las_piedras_2026-03-13 | 相似度: 73.3 分


In [ ]:
from rapidfuzz import fuzz

def calculate_custom_similarity(id1: str, id2: str) -> float:
    # 觀察你的 ID 格式，最後 10 個字元固定是日期 "YYYY-MM-DD"
    # 我們把字串切成「球隊部分」和「日期部分」
    date1 = id1[-10:]
    date2 = id2[-10:]
    
    # =========================================
    #  權重規則 1：時間不同，直接歸 0！ (Blocking)
    # =========================================
    if date1 != date2:
        return 0.0
        
    # 把前面的球隊部分切出來，並把底線換成空白，方便演算法切單字
    # id1[:-11] 代表去掉 "_2026-03-13" 這 11 個字元
    teams1 = id1[:-11].replace("_", " ") 
    teams2 = id2[:-11].replace("_", " ") 
    
    # =========================================
    #  權重規則 2：名子有重疊加分！ (Token Set Ratio)
    # =========================================
    # token_set_ratio 非常聰明，只要雙方有高度重疊的單字，分數就會暴衝
    score = fuzz.token_set_ratio(teams1, teams2)
    
    return score

# ---  測試你的真實案例 ---
if __name__ == "__main__":
    target_id = "independiente_medellín_ca_juventud_2026-03-13"
    
    candidates = [
        # 1. 你的真實案例 (名字有重疊，日期相同)
        "independiente_medellín_juventud_de_las_piedras_2026-03-13",
        
        # 2. 測試日期不同 (差一天)
        "independiente_medellín_cdp_junior_2026-03-13",
        
        # 3. 測試完全不同的比賽
        "arsenal_chelsea_2026-03-13"
    ]
    
    print("="*60)
    print(f" 目標比賽: {target_id}")
    print("="*60)
    
    for cand in candidates:
        score = calculate_custom_similarity(target_id, cand)
        print(f"候選: {cand}")
        if score >80:
            print(f"  [過關] 名字高度重疊，分數: {score:.1f} 分\n")
        else:
            print(f"  [淘汰]  日期不同或完全不配，分數:{score:.1f} 分\n")

 目標比賽: independiente_medellín_ca_juventud_2026-03-13
候選: independiente_medellín_juventud_de_las_piedras_2026-03-13
  [過關] 名字高度重疊，分數: 95.4 分

候選: independiente_medellín_cdp_junior_2026-03-13
  [過關] 名字高度重疊，分數: 83.6 分

候選: arsenal_chelsea_2026-03-13
  [淘汰]  日期不同或完全不配，分數:24.5 分



In [11]:
pip install rapidfuzz

  Obtaining dependency information for rapidfuzz from https://files.pythonhosted.org/packages/cf/99/5fa23e204435803875daefda73fd61baeabc3c36b8fc0e34c1705aab8c7b/rapidfuzz-3.14.3-cp311-cp311-win_amd64.whl.metadata
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
    --------------------------------------- 0.0/1.5 MB 660.6 kB/s eta 0:00:03
   -- ------------------------------------- 0.1/1.5 MB 1.2 MB/s eta 0:00:02
   ------ --------------------------------- 0.3/1.5 MB 2.0 MB/s eta 0:00:01
   ------------------- -------------------- 0.8/1.5 MB 4.0 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 7.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
import json

# 開啟並讀取 JSON 檔案
with open('matches_db_debug.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# 取得所有的 key
keys = data.keys()

# 轉換成列表 (list) 方便查看或後續操作
key_list = list(keys)
print(key_list)

['independiente_medellín_juventud_de_las_piedras_2026-03-13', 'west_ham_manchester_city_2026-03-15', 'burnley_bournemouth_2026-03-14', 'chelsea_newcastle_2026-03-15', 'sunderland_brighton_and_hove_albion_2026-03-14', 'arsenal_everton_2026-03-15', 'gil_vicente_alverca_2026-03-15', 'arouca_sl_benfica_2026-03-15', 'vitoria_guimaraes_famalicão_2026-03-14', 'manchester_aston_villa_2026-03-15', 'liverpool_tottenham_hotspur_2026-03-16', 'porto_moreirense_2026-03-16', 'avs_futebol_sad_santa_clara_2026-03-15', 'club_do_remo_fluminense_2026-03-13', 'sao_paulo_chapecoense_2026-03-13', 'cr_vasco_da_gama_palmeiras_sp_2026-03-13', 'panathinaikos_real_betis_2026-03-13', 'krc_genk_sport_club_freiburg_2026-03-13', 'lech_poznan_shakhtar_donetsk_2026-03-13', 'lille_aston_villa_2026-03-13', 'nk_cm_celje_aek_athens_2026-03-13', 'bologna_as_roma_2026-03-13', 'az_alkmaar_ac_sparta_praha_2026-03-13', 'sk_sigma_olomouc_fsv_mainz_2026-03-13', 'vfb_stuttgart_porto_2026-03-13', 'nottingham_forest_midtjylland_2026

In [22]:
len(key_list)

1315

In [24]:
count=0
for key1 in key_list:
    for key2 in key_list:
        if key1 != key2:
            score = calculate_custom_similarity(key1, key2)
            if score > 80:
                count+=1
                print(f"可能重複的比賽對: \n  - {key1}\n  - {key2}\n  相似度: {score:.1f} 分\n")

可能重複的比賽對: 
  - independiente_medellín_juventud_de_las_piedras_2026-03-13
  - independiente_medellín_ca_juventud_2026-03-13
  相似度: 95.4 分

可能重複的比賽對: 
  - sunderland_brighton_and_hove_albion_2026-03-14
  - sunderland_brighton_&_hove_albion_2026-03-14
  相似度: 96.9 分

可能重複的比賽對: 
  - sunderland_brighton_and_hove_albion_2026-03-14
  - sunderland_brighton_2026-03-14
  相似度: 100.0 分

可能重複的比賽對: 
  - avs_futebol_sad_santa_clara_2026-03-15
  - avs_futebol_cd_santa_clara_2026-03-15
  相似度: 94.3 分

可能重複的比賽對: 
  - nacional_madeira_estoril_2026-03-15
  - cd_nacional_estoril_praia_2026-03-15
  相似度: 81.6 分

可能重複的比賽對: 
  - gremio_red_bull_bragantino_2026-03-13
  - grêmio_fbpa_red_bull_bragantino_2026-03-13
  相似度: 87.7 分

可能重複的比賽對: 
  - wuhan_three_towns_dalian_yingbo_2026-03-13
  - wuhan_san_zhen_dalian_yingbo_2026-03-13
  相似度: 80.9 分

可能重複的比賽對: 
  - borussia_dortmund_augsburg_2026-03-14
  - bv_borussia_09_dortmund_augsburg_2026-03-14
  相似度: 100.0 分

可能重複的比賽對: 
  - tsg_1899_hoffenheim_vfl_wolfsburg_2026-03

In [103]:
import requests
event_id="250467"
gamma_url = f"https://gamma-api.polymarket.com/events/{event_id}"
print(f"正在呼叫 Gamma API 取得市場結構...")

try:
    response = requests.get(gamma_url, timeout=10)
    response.raise_for_status()
    event_data = response.json()
except requests.exceptions.RequestException as e:
    print(f"❌ 呼叫 Gamma API 失敗: {e}")

正在呼叫 Gamma API 取得市場結構...


In [104]:
event_data

{'id': '250467',
 'ticker': 'uel-ast4-lil-2026-03-19',
 'slug': 'uel-ast4-lil-2026-03-19',
 'title': 'Aston Villa FC vs. Lille OSC',
 'description': 'This event is for the upcoming UEFA Europa League game, scheduled for Thursday, March 19, 2026 between Aston Villa FC and Lille OSC.',
 'resolutionSource': 'https://www.uefa.com/',
 'startDate': '2026-03-06T19:36:27.812988Z',
 'creationDate': '2026-03-06T19:36:27.812984Z',
 'endDate': '2026-03-19T20:00:00Z',
 'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/uel.png',
 'icon': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/uel.png',
 'active': True,
 'closed': False,
 'archived': False,
 'new': False,
 'featured': False,
 'restricted': True,
 'liquidity': 975205.2431,
 'volume': 2829.225518,
 'openInterest': 0,
 'createdAt': '2026-03-06T19:30:02.663534Z',
 'updatedAt': '2026-03-17T14:08:27.326668Z',
 'competitive': 0.9928268261808434,
 'volume24hr': 1118.461485,
 'volume1wk': 2829.2255179999997,
 'volume1mo': 2829.2255

In [105]:
import time
token_id="62033066181846619602217677397394431047823124200859036296834340893601132584539"
clob_url = "https://clob.polymarket.com/book"
                
try:
    time.sleep(0.2) # 禮貌性暫停，避免被鎖 IP
    
    clob_res = requests.get(clob_url, params={"token_id": token_id}, timeout=10)
    clob_res.raise_for_status()
    book_data = clob_res.json()
except requests.exceptions.RequestException as e:
    print(f" 呼叫 CLOB API 失敗: {e}")

In [106]:
book_data


{'market': '0x9434369bf8eddd8e3a8e332b13d52a9547b55fc54ad3fa09b91d7191a8b7fc7b',
 'asset_id': '62033066181846619602217677397394431047823124200859036296834340893601132584539',
 'timestamp': '1773756776840',
 'hash': '7a9931b5ded5e8b45929df111081122303fd024f',
 'bids': [{'price': '0.01', 'size': '684.92'},
  {'price': '0.02', 'size': '155'},
  {'price': '0.03', 'size': '81'},
  {'price': '0.04', 'size': '14.18'},
  {'price': '0.25', 'size': '5000'},
  {'price': '0.32', 'size': '5000'},
  {'price': '0.35', 'size': '46992'},
  {'price': '0.38', 'size': '39824'},
  {'price': '0.39', 'size': '5015'},
  {'price': '0.41', 'size': '32992'},
  {'price': '0.44', 'size': '26468'},
  {'price': '0.45', 'size': '83891.26'},
  {'price': '0.46', 'size': '5036.52'},
  {'price': '0.47', 'size': '20160'},
  {'price': '0.48', 'size': '137'},
  {'price': '0.49', 'size': '16.32'},
  {'price': '0.5', 'size': '14000'},
  {'price': '0.51', 'size': '77191.63'},
  {'price': '0.52', 'size': '325'},
  {'price': '0.

In [26]:
bids = book_data.get("bids", [])
asks = book_data.get("asks", [])

In [28]:
bids[-1]

{'price': '0.28', 'size': '3792.12'}

In [29]:
asks[-1]

{'price': '0.29', 'size': '173.35'}

In [79]:
market_hash = "0x3c9f3ad8322d372bfb3a0139ebd37bf51f9ac9903960642d934538ce488d30e2"
url = "https://api.sx.bet/orders"
params = {
    "marketHashes": market_hash, # 注意：官方文件寫的是 marketHashes (有s)
}
headers = {"User-Agent": "Mozilla/5.0"}

try:
    response = requests.get(url, params=params, headers=headers, timeout=10)
    response.raise_for_status()
    
    data = response.json()
    orders = data.get("data", [])
    
    print(f" 成功連線！這場比賽目前共有 {len(orders)} 筆掛單等待被吃。\n")
    
    if not orders:
        print(" 目前沒有掛單。")
        
        
    outcome_1_odds = [] 
    outcome_2_odds = [] 
    
    for order in orders:
        # 1. 計算剩餘數量 (區塊鏈原始單位)
        total_bet_size = float(order.get("totalBetSize", 0))
        fill_amount = float(order.get("fillAmount", 0))
        remaining_raw = total_bet_size - fill_amount
        
        if remaining_raw <= 0:
            continue # 已經被買光的單就跳過
            
        # 將數量除以 10^6 轉換成大概的 USDC 美金數量 (方便人類閱讀)
        remaining_usdc = remaining_raw / (10**6)
        
        # 2. 核心數學：計算 Taker (也就是我們) 的隱含勝率成本
        percentage_odds_raw = float(order.get("percentageOdds", 0))
        taker_implied_prob = 1 - (percentage_odds_raw / (10**20))
        taker_implied_prob=1/taker_implied_prob
        # 3. 分類邏輯：Maker 賭一邊，我們接單就是賭另一邊
        is_maker_outcome_one = order.get("isMakerBettingOutcomeOne", False)
        
        order_info = {
            "price": taker_implied_prob, 
            "size": remaining_usdc,
            "base_token": order.get("baseToken", "")
        }
        if is_maker_outcome_one:
            # Maker 買 Outcome 1，我們接單是買 Outcome 2
            outcome_2_odds.append(order_info)
        else:
            # Maker 買 Outcome 2，我們接單是買 Outcome 1
            outcome_1_odds.append(order_info)
except requests.exceptions.RequestException as e:
    print(f" 請求失敗: {e}")

 成功連線！這場比賽目前共有 0 筆掛單等待被吃。

 目前沒有掛單。


In [108]:
market_hash = "0x36a4a839dd5383f49d4b964419843bdbdf31d03de6c6e881f2b7835050ee70b0"
url = "https://api.sx.bet/orders"
params = {
    "marketHashes": market_hash, # 注意：官方文件寫的是 marketHashes (有s)
}
headers = {"User-Agent": "Mozilla/5.0"}

try:
    response = requests.get(url, params=params, headers=headers, timeout=10)
    response.raise_for_status()
    
    data = response.json()
    orders = data.get("data", [])
    
    print(f" 成功連線！這場比賽目前共有 {len(orders)} 筆掛單等待被吃。\n")
    
    if not orders:
        print(" 目前沒有掛單。")
        
        
    outcome_1_odds = [] 
    outcome_2_odds = [] 
    
    for order in orders:
        # 1. 計算剩餘數量 (區塊鏈原始單位)
        total_bet_size = float(order.get("totalBetSize", 0))
        fill_amount = float(order.get("fillAmount", 0))
        remaining_raw = total_bet_size - fill_amount
        
        if remaining_raw <= 0:
            continue # 已經被買光的單就跳過
            
        # 這是 Maker (掛單者) 剩餘的本金
        remaining_maker_usdc = remaining_raw / (10**6)
        
        # 2. 核心數學：計算雙方的隱含勝率
        percentage_odds_raw = float(order.get("percentageOdds", 0))
        maker_implied_prob = percentage_odds_raw / (10**20)  # 掛單者的勝率
        taker_implied_prob = 1 - maker_implied_prob          # 我們的勝率 (Price)
        
        # 避免除以零的錯誤
        if maker_implied_prob <= 0 or taker_implied_prob <= 0:
            continue
            
        # 算網頁常見的小數點賠率 (例如 2.50)
        taker_decimal_odds = 1 / taker_implied_prob 
        
        # 🌟 3. 將 Maker 的本金，換算成網頁上顯示的「我們可下注的最大金額 (Taker Size)」
        taker_size = remaining_maker_usdc * (taker_implied_prob / maker_implied_prob)
        
        # 4. 分類邏輯：Maker 賭一邊，我們接單就是賭另一邊
        is_maker_outcome_one = order.get("isMakerBettingOutcomeOne", False)
        
        order_info = {
            "implied_prob": taker_implied_prob,  # 用來算套利的成本 (0~1)
            "decimal_odds": taker_decimal_odds,  # 用來給人看的賠率 (大於1)
            "size": taker_size,                  #  這就是網頁上顯示的可下注金額！
            "base_token": order.get("baseToken", "")
        }
        
        if is_maker_outcome_one:
            # Maker 買 Outcome 1，我們接單是買 Outcome 2
            outcome_2_odds.append(order_info)
        else:
            # Maker 買 Outcome 2，我們接單是買 Outcome 1
            outcome_1_odds.append(order_info)
except requests.exceptions.RequestException as e:
    print(f" 請求失敗: {e}")




 成功連線！這場比賽目前共有 12 筆掛單等待被吃。



In [60]:
outcome_1_odds

[{'implied_prob': 0.45875,
  'decimal_odds': 2.1798365122615806,
  'size': 96.14043879907621,
  'base_token': '0x6629Ce1Cf35Cc1329ebB4F63202F3f197b3F050B'},
 {'implied_prob': 0.47250000000000003,
  'decimal_odds': 2.1164021164021163,
  'size': 359.9867772511849,
  'base_token': '0x6629Ce1Cf35Cc1329ebB4F63202F3f197b3F050B'},
 {'implied_prob': 0.4575,
  'decimal_odds': 2.1857923497267757,
  'size': 81.6416129032258,
  'base_token': '0x6629Ce1Cf35Cc1329ebB4F63202F3f197b3F050B'},
 {'implied_prob': 0.45625000000000004,
  'decimal_odds': 2.191780821917808,
  'size': 12.586206896551728,
  'base_token': '0x6629Ce1Cf35Cc1329ebB4F63202F3f197b3F050B'}]

In [61]:
outcome_2_odds

[{'implied_prob': 0.57125,
  'decimal_odds': 1.75054704595186,
  'size': 115.50241982507289,
  'base_token': '0x6629Ce1Cf35Cc1329ebB4F63202F3f197b3F050B'},
 {'implied_prob': 0.5700000000000001,
  'decimal_odds': 1.7543859649122806,
  'size': 80.44953488372093,
  'base_token': '0x6629Ce1Cf35Cc1329ebB4F63202F3f197b3F050B'},
 {'implied_prob': 0.56125,
  'decimal_odds': 1.7817371937639197,
  'size': 208.74022792022794,
  'base_token': '0x6629Ce1Cf35Cc1329ebB4F63202F3f197b3F050B'},
 {'implied_prob': 0.60375,
  'decimal_odds': 1.6563146997929605,
  'size': 359.99498422712935,
  'base_token': '0x6629Ce1Cf35Cc1329ebB4F63202F3f197b3F050B'}]

In [62]:
order_info

{'implied_prob': 0.45625000000000004,
 'decimal_odds': 2.191780821917808,
 'size': 12.586206896551728,
 'base_token': '0x6629Ce1Cf35Cc1329ebB4F63202F3f197b3F050B'}

In [54]:
print(f"賠率: {order_info['decimal_odds']:.2f} | 網頁顯示可下注量 (Size): ${order_info['size']:.2f}")

賠率: 2.19 | 網頁顯示可下注量 (Size): $81.64


In [67]:
parent_slug="serie-a-pisa-vs-cagliari-mar-15-2026-1772355602827"
headers = {"Accept": "application/json", "User-Agent": "Mozilla/5.0"}

# ==========================================
#  步驟一：取得母賽事結構，萃取子盤口 Slug
# ==========================================
market_url = f"https://api.limitless.exchange/markets/{parent_slug}"
print(f" [1/2] 正在獲取賽事結構...")

try:
    res = requests.get(market_url, headers=headers, timeout=10)
    res.raise_for_status()
    parent_data = res.json()
    
    # 檢查這是不是群組市場
    market_type = parent_data.get("marketType", "")
    if market_type != "group":
        print(f" 這似乎不是群組市場 (類型: {market_type})，可能不適用本程式邏輯。")
        
    markets = parent_data.get("markets", [])
    if not markets:
        print(" 找不到子盤口 (Markets)。")
        
        
    print(f" 成功找到 {len(markets)} 個子盤口，準備分別獲取訂單簿...\n")
    print("-" * 50)
    
    # ==========================================
    #  步驟二：遍歷子盤口，獲取各自的 Orderbook
    # ==========================================
    for market in markets:
        title = market.get("title", "未知選項")
        child_slug = market.get("slug")
        
        if not child_slug:
            print(f" 選項 [{title}] 沒有專屬 slug，跳過。")
            continue
            
        print(f" 選項: {title} (Slug: {child_slug})")
        
        # 呼叫 Orderbook API
        ob_url = f"https://api.limitless.exchange/markets/{child_slug}/orderbook"
        
        try:
            time.sleep(0.2) # 禮貌性暫停
            ob_res = requests.get(ob_url, headers=headers, timeout=10)
            ob_res.raise_for_status()
            ob_data = ob_res.json()
            
            bids = ob_data.get("bids", [])
            asks = ob_data.get("asks", [])
        except requests.exceptions.RequestException as e:
            print(f"  ❌ 無法獲取 Orderbook: {e}")
            continue
except requests.exceptions.RequestException as e:
    print(f" ❌ 無法獲取賽事結構: {e}")


 [1/2] 正在獲取賽事結構...
 成功找到 3 個子盤口，準備分別獲取訂單簿...

--------------------------------------------------
 選項: Pisa (Slug: pisa-1772355602833)
 選項: Cagliari (Slug: cagliari-1772355602843)
 選項: Draw (Slug: draw-1772355602860)


In [68]:
ob_data

{'bids': [{'price': 0.32, 'size': 13500000000, 'side': 'BUY'},
  {'price': 0.31, 'size': 3500000000, 'side': 'BUY'},
  {'price': 0.305, 'size': 400000000, 'side': 'BUY'},
  {'price': 0.295, 'size': 100000000, 'side': 'BUY'}],
 'asks': [{'price': 0.33, 'size': 17000000000, 'side': 'SELL'},
  {'price': 0.35, 'size': 400000000, 'side': 'SELL'},
  {'price': 0.355, 'size': 100000000, 'side': 'SELL'}],
 'tokenId': '99963312708948001263281629084055988886434934821848492694729133929296074592210',
 'adjustedMidpoint': 0.325,
 'midpoint': 0.325,
 'maxSpread': '0.035',
 'minSize': '100000000',
 'lastTradePrice': None}

In [84]:
from rapidfuzz import fuzz
def calculate_custom_similarity(id1: str, id2: str) -> float:
    # 觀察你的 ID 格式，最後 10 個字元固定是日期 "YYYY-MM-DD"
    # 我們把字串切成「球隊部分」和「日期部分」
    date1 = id1[-10:]
    date2 = id2[-10:]
    
    # =========================================
    #  權重規則 1：時間不同，直接歸 0！ (Blocking)
    # =========================================
    if date1 != date2:
        return 0.0
        
    # 把前面的球隊部分切出來，並把底線換成空白，方便演算法切單字
    # id1[:-11] 代表去掉 "_2026-03-13" 這 11 個字元
    teams1 = id1[:-11].replace("_", " ") 
    teams2 = id2[:-11].replace("_", " ") 
    
    # =========================================
    #  權重規則 2：名子有重疊加分！ (Token Set Ratio)
    # =========================================
    # token_set_ratio 非常聰明，只要雙方有高度重疊的單字，分數就會暴衝
    score = fuzz.token_set_ratio(teams1, teams2)
    
    return score

In [95]:
a="brentford_wolves_2026-03-16"
b="brentford_wolverhampton_wanderers_2026-03-16"
ans=calculate_custom_similarity(a,b)

In [96]:
ans

72.0

In [93]:
a="olympique_lyonnais_as_monaco_2026-03-22"
b="lyon_monaco_2026-03-22"
ans=calculate_custom_similarity(a,b)
ans


70.58823529411765

In [98]:

a="real_madrid_club_atlético_de_madrid_2026-03-22"

b="real_madrid_atletico_madrid_2026-03-22"
ans=calculate_custom_similarity(a,b)
ans

79.16666666666667

In [102]:
a="rc_celta_de_vigo_deportivo_alavés_2026-03-22"
b="celta_vigo_alaves_2026-03-22"
ans=calculate_custom_similarity(a,b)
ans

74.07407407407408

In [101]:
a="sport_lisboa_e_benfica_vitória_sc_2026-03-21"
b="sl_benfica_vitoria_guimaraes_2026-03-21"
ans=calculate_custom_similarity(a,b)
ans

65.57377049180329

In [100]:
a="stade_rennais_1901_metz_2026-03-22"
b="rennes_metz_2026-03-22"
ans=calculate_custom_similarity(a,b)
ans

58.8235294117647

In [109]:
import requests
import json

def fetch_usdc_tokens():
    url = "https://api.sx.bet/metadata"
    print(f"🔍 正在向 {url} 請求平台元數據...\n")
    
    try:
        # 發送 GET 請求
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        
        # 解析 JSON
        metadata = response.json()
        
        # SX Bet 的 metadata 通常包在 "data" 裡面，裡面會有 "tokens" 或 "currencies" 列表
        # 我們直接從裡面篩選出 symbol 是 USDC 的代幣
        tokens = metadata.get("data", {}).get("tokens", [])
        
        if not tokens:
            print("⚠️ 在 metadata 中找不到 tokens 列表，請檢查完整 JSON 結構。")
            return

        print("🎯 找到以下 USDC 代幣地址：")
        print("-" * 60)
        
        usdc_found = False
        for token in tokens:
            symbol = token.get("symbol", "").upper()
            
            if "USDC" in symbol:
                usdc_found = True
                address = token.get("address")
                chain_id = token.get("chainId")
                name = token.get("name")
                
                # 簡單判斷一下是哪條鏈 (Arbitrum 通常是 42161, Polygon 是 137)
                chain_name = "Arbitrum" if chain_id == 42161 else "Polygon" if chain_id == 137 else f"Chain {chain_id}"
                
                print(f"🔹 網路 (Chain): {chain_name}")
                print(f"   名稱 (Name): {name}")
                print(f"   地址 (Address): {address}")
                print("-" * 60)
                
        if not usdc_found:
            print("❌ 找不到任何 USDC 的地址。")
            
    except Exception as e:
        print(f"❌ 請求失敗: {e}")

if __name__ == "__main__":
    fetch_usdc_tokens()

🔍 正在向 https://api.sx.bet/metadata 請求平台元數據...

⚠️ 在 metadata 中找不到 tokens 列表，請檢查完整 JSON 結構。


In [2]:
pip uninstall centrifuge centrifuge-python -y

Found existing installation: centrifuge-python 0.4.3
Uninstalling centrifuge-python-0.4.3:
  Successfully uninstalled centrifuge-python-0.4.3
Note: you may need to restart the kernel to use updated packages.


In [ ]:
pip install centrifuge-python aiohttp requests

  Obtaining dependency information for centrifuge-python from https://files.pythonhosted.org/packages/72/75/bc482eb1ea6b4c2020d2453e6abde9f4b790a0e1cd23071163e9b33455e5/centrifuge_python-0.4.3-py3-none-any.whl.metadata
  Using cached centrifuge_python-0.4.3-py3-none-any.whl.metadata (5.6 kB)
Using cached centrifuge_python-0.4.3-py3-none-any.whl (26 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install python-socketio aiohttp

  Obtaining dependency information for python-socketio from https://files.pythonhosted.org/packages/07/c7/deb8c5e604404dbf10a3808a858946ca3547692ff6316b698945bb72177e/python_socketio-5.16.1-py3-none-any.whl.metadata
  Obtaining dependency information for bidict>=0.21.0 from https://files.pythonhosted.org/packages/99/37/e8730c3587a65eb5645d4aba2d27aae48e8003614d6aaf15dda67f702f1f/bidict-0.23.1-py3-none-any.whl.metadata
  Obtaining dependency information for python-engineio>=4.11.0 from https://files.pythonhosted.org/packages/aa/54/0cce26da03a981f949bb8449c9778537f75f5917c172e1d2992ff25cb57d/python_engineio-4.13.1-py3-none-any.whl.metadata
  Obtaining dependency information for simple-websocket>=0.10.0 from https://files.pythonhosted.org/packages/52/59/0782e51887ac6b07ffd1570e0364cf901ebc36345fea669969d2084baebb/simple_websocket-1.1.0-py3-none-any.whl.metadata
  Obtaining dependency information for wsproto from https://files.pythonhosted.org/packages/a4/f5/10b68b7b1544245097b2a1b8238f66f


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
api_key2 = os.environ.get("SX_bet")
print(f"🔑 目前的 API Key 是: {api_key2}")

AttributeError: module 'os' has no attribute 'env'

In [7]:
from dotenv import load_dotenv

In [6]:
pip install python-dotenv

  Obtaining dependency information for python-dotenv from https://files.pythonhosted.org/packages/0b/d7/1959b9648791274998a9c3526f6d0ec8fd2233e4d4acce81bbae76b44b2a/python_dotenv-1.2.2-py3-none-any.whl.metadata
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import sys
import os


from platforms.limitless import LimitlessAPI
from utils.team_mapping import TeamNameMapper
mapper = TeamNameMapper()
lim_api = LimitlessAPI(mapper)
ob = lim_api.get_orderbook(market_id="real-madrid-1774342805684" , selection='madari')
            
if ob.best_bid:
    bid_decimal = 1.0 / ob.best_bid.price if ob.best_bid.price > 0 else 0
    print(f"      最高買價 (Bid): 隱含機率 {ob.best_bid.price:.4f} (賠率 {bid_decimal:.2f}) | 可用數量: ${ob.best_bid.size:,.2f} <-- 脫手價")
else:
    print(f"      最高買價 (Bid): 目前沒有買單")
    
if ob.best_ask:
    ask_decimal = 1.0 / ob.best_ask.price if ob.best_ask.price > 0 else 0
    print(f"      最低賣價 (Ask): 隱含機率 {ob.best_ask.price:.4f} (賠率 {ask_decimal:.2f}) | 可用數量: ${ob.best_ask.size:,.2f} <-- 買入成本")
else:
    print(f"      最低賣價 (Ask): 目前沒有賣單")

      最高買價 (Bid): 隱含機率 0.3267 (賠率 3.06) | 可用數量: $26,500.00 <-- 脫手價
      最低賣價 (Ask): 隱含機率 0.3502 (賠率 2.86) | 可用數量: $26,500.00 <-- 買入成本


In [24]:
def get_buy_fee(p: float) -> float:
    """Taker Buy Fee: 0~0.5 為 3%，0.5~1.0 線性下降至 0.03%"""
    if p <= 0.5:
        return 0.03
    else:
        progress = (p - 0.5) / 0.5
        return 0.03 - (0.039869281045751624 * progress)

In [30]:
print(get_buy_fee(0.833))

0.003447058823529421


In [22]:

x=(0.03-0.0178)*0.5/(0.653 - 0.5)

In [10]:
import time

# 匯入各平台 API
from platforms import AVAILABLE_PLATFORMS

# 匯入核心工具 (依據你的資料夾結構，若放在 utils 請自行更改)

from utils.team_mapping import TeamNameMapper

# 匯入配對引擎
# 假設你已經將兩階段配對邏輯存放在 core/matcher.py 中
from core.matcher import MatchEngine 


print("  Total Search")

# 步驟 1：初始化系統工具與平台

mapper = TeamNameMapper()

# 初始化配對引擎：門檻 70 分，至少需要 2 個平台重疊
match_engine = MatchEngine(threshold=70.0, min_platforms=2)

# 註冊我們要掃描的平台
mapper = TeamNameMapper()

# 動態實例化所有在 __init__.py 註冊過的平台！
platforms = [PlatformClass(mapper) for PlatformClass in AVAILABLE_PLATFORMS]


# 步驟 2：第一階段 - 盤前靜態資料收集

all_events = []
for api in platforms:
    try:
        events = api.get_matches()
        all_events.extend(events)
    except Exception as e:
        print(f" {api.name} 獲取賽事失敗: {e}")


# 步驟 3：安檢過濾 - 只保留安全的套利標的

# 1. 確保是獨贏盤 (moneyline)
# 2. 確保我們有成功解析出子盤口的鑰匙 (token_mapping)
moneyline_events = [
    e for e in all_events 
    if e.market_type == "moneyline" and e.raw_data.get("token_mapping")
]

print(f"\n 總計獲取 {len(all_events)} 場比賽。")
print(f" 過濾後剩下 {len(moneyline_events)} 場高品質 Moneyline (獨贏盤) 賽事準備進行配對。")


# 步驟 4：啟動 AI 配對引擎，尋找 Overlap

overlapping_matches = match_engine.match_events(moneyline_events)

  Total Search

[SX_Bet] 開始獲取足球賽事資料 (支援自動分頁並聚合盤口)...
[SX_Bet] 正在抓取第 1 頁...
[SX_Bet] 正在抓取第 2 頁...
[SX_Bet] 正在抓取第 3 頁...
[SX_Bet] 正在抓取第 4 頁...
[SX_Bet] 正在抓取第 5 頁...
[SX_Bet] 正在抓取第 6 頁...
[SX_Bet] 正在抓取第 7 頁...
[SX_Bet] 正在抓取第 8 頁...
[SX_Bet] 正在抓取第 9 頁...
[SX_Bet] 正在抓取第 10 頁...
[SX_Bet] 正在抓取第 11 頁...
[SX_Bet] 正在抓取第 12 頁...
[SX_Bet] 正在抓取第 13 頁...
[SX_Bet] 成功合併出 94 場標準化賽事！

[Polymarket] 開始使用 `offset` 參數自動翻頁抓取資料...
[Polymarket] 正在抓取第 1 頁 (Offset: 0)...
[Polymarket] 正在抓取第 2 頁 (Offset: 500)...
[Polymarket] 正在抓取第 3 頁 (Offset: 1000)...
[Polymarket] 正在抓取第 4 頁 (Offset: 1500)...
[Polymarket] 正在抓取第 5 頁 (Offset: 2000)...
[Polymarket] 正在抓取第 6 頁 (Offset: 2500)...
[Polymarket] 正在抓取第 7 頁 (Offset: 3000)...
[Polymarket] 已經到達最後一頁，抓取完畢！

[Polymarket] 成功獲取 1519 場標準化賽事！

[Limitless] 開始獲取足球賽事資料...
[Limitless] 正在抓取第 1 頁...
[Limitless] 正在抓取第 2 頁...
[Limitless] 正在抓取第 3 頁...
[Limitless] 正在抓取第 4 頁...
[Limitless] 正在抓取第 5 頁...
[Limitless] 正在抓取第 6 頁...
[Limitless] 成功獲取 139 場標準化賽事！

 總計獲取 1752 場比賽。
 過濾後剩下 1752 場高品質 Moneyl

In [11]:
overlapping_matches

[('cusco_cr_flamengo_2026-04-09',
  [StandardEvent(home_team='cusco', away_team='cr_flamengo', start_time=datetime.datetime(2026, 4, 9, 0, 30, tzinfo=datetime.timezone.utc), platform='SX_Bet', platform_event_id='cusco_cr_flamengo_2026-04-09', market_type='moneyline', market_name='Cusco vs CR Flamengo', raw_data={'token_mapping': {'Draw': '0xd6ad7a2b46fe1b1b0ce37798c2da787fc5688c5afa76d41527a80aa8eb9c6e90', 'Not Draw': '0xd6ad7a2b46fe1b1b0ce37798c2da787fc5688c5afa76d41527a80aa8eb9c6e90', 'Home': '0xa54d9cb1e5d074989dd202c0debf1e69dc737c71bf3ff92a9549e26541ac5abf', 'Not Home': '0xa54d9cb1e5d074989dd202c0debf1e69dc737c71bf3ff92a9549e26541ac5abf', 'Away': '0x7252e463041310b5e7f7971b575a1c64e7cffd92592b82039ca857c69dfd3d2a', 'Not Away': '0x7252e463041310b5e7f7971b575a1c64e7cffd92592b82039ca857c69dfd3d2a'}}),
   StandardEvent(home_team='cusco', away_team='cr_flamengo', start_time=datetime.datetime(2026, 4, 9, 0, 30, tzinfo=datetime.timezone.utc), platform='Polymarket', platform_event_id='314